In [2]:
import pandas as pd
import numpy as np

# 1. 读取数据 (这里一定要写上完整的 .xlsx 后缀！)
file_name = 'BA_overseas_data.xlsx'
df = pd.read_excel(file_name)
print(f"读取成功！原始数据形状: {df.shape}")
print(f"包含列名: {df.columns.tolist()}\n")

# 2. 数据去重
initial_count = len(df)
df = df.drop_duplicates(subset=['Job_Url'], keep='first')
print(f"去重完成: 删除了 {initial_count - len(df)} 条重复记录，剩余 {len(df)} 条。\n")

# 3. 缺失值处理
df['Education'] = df['Education'].fillna('Not Specified')
df['Industry'] = df['Industry'].fillna('Unknown')
df['Experience_Min'] = df['Experience_Min'].fillna('Not Specified')
df['Experience_Max'] = df['Experience_Max'].fillna('Not Specified')

df['Salary_Min'] = pd.to_numeric(df['Salary_Min'], errors='coerce')
df['Salary_Max'] = pd.to_numeric(df['Salary_Max'], errors='coerce')

# 4. 处理异常值 (剔除那条澳洲异常数据)
df = df[~((df['Salary_Min'] == 85000) & (df['Salary_Max'] == 10000))]
print(f"清理异常值后，剩余 {len(df)} 条数据。\n")

# 5. 技能列标准化
# 注意：A同学的新数据是 'ai_skill_explicit'，不是 'AI_tools'
skill_cols = ['Excel', 'SQL', 'Python', 'R', 'Tableau', 'Power_BI', 'ai_skill_explicit']
for col in skill_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)
        df[col] = df[col].astype(int)
print(f"技能列已标准化为 0/1: {skill_cols}\n")

# 6. 日期格式化 (修复了 erors 的拼写错误)
df['Date_Posted'] = pd.to_datetime(df['Date_Posted'], errors='coerce')
df['Date_Collected'] = pd.to_datetime(df['Date_Collected'], errors='coerce')

# 7. 保存清洗后的数据
cleaned_file_name = 'pilot_data_cleaned.csv'
df.to_csv(cleaned_file_name, index=False)
print(f"清洗后的数据已保存为: {cleaned_file_name}")
print(f"最终清洗数据形状: {df.shape}")

# 8. 快速检查前 5 行
df.head()

读取成功！原始数据形状: (199, 26)
包含列名: ['Job_id', 'Country', 'City', 'Job_Title', 'Company_Name', 'Industry', 'Source', 'Job_Url', 'Date_Posted', 'Date_Collected', 'Salary_Min', 'Salary_Max', 'Salary_Currency', 'Experience_Min', 'Experience_Max', 'Education', 'Excel', 'SQL', 'Python', 'R', 'Tableau', 'Power_BI', 'ai_skill_explicit', 'ai_skill_level', 'JD_text', 'Collector']

去重完成: 删除了 2 条重复记录，剩余 197 条。

清理异常值后，剩余 196 条数据。

技能列已标准化为 0/1: ['Excel', 'SQL', 'Python', 'R', 'Tableau', 'Power_BI', 'ai_skill_explicit']

清洗后的数据已保存为: pilot_data_cleaned.csv
最终清洗数据形状: (196, 26)


,Job_id,Country,City,Job_Title,Company_Name,Industry,Source,Job_Url,Date_Posted,Date_Collected,...,Excel,SQL,Python,R,Tableau,Power_BI,ai_skill_explicit,ai_skill_level,JD_text,Collector
0,BA001,United States,"Atlanta, GA",Staff Business Analyst,TriNet,Technology,LinkedIn,https://www.linkedin.com/jobs/view/staff-busin...,2026-07-16,2026-09-15,...,0,0,0,0,0,0,0,0.0,TriNet is a leading provider of comprehensive ...,A
1,BA002,United States,"Reston, VA",Business Analyst (ServiceNow) - (High Level Cl...,ICF,Technology,LinkedIn,https://www.linkedin.com/jobs/view/business-an...,2026-09-14,2026-09-15,...,0,0,0,0,0,0,0,0.0,Description ICF is seeking a motivated Busines...,A
2,BA003,United States,"Miami, FL","Business Analyst, Fuse Billing Operations",Amazon,Technology,LinkedIn,https://www.linkedin.com/jobs/view/business-an...,2026-09-09,2026-09-15,...,1,1,0,0,1,0,0,0.0,Description The Amazon Mobile Business Develop...,A
3,BA004,United States,"Boston, MA",Systems Business Analyst Sr.,Brown Brothers Harriman,Technology,LinkedIn,https://www.linkedin.com/jobs/view/systems-bus...,2026-09-17,2026-09-15,...,1,1,0,0,0,0,0,0.0,"At BBH, Partnership is more than a form of own...",A
4,BA005,United States,"Las Vegas, NV",Sr Business Analyst,Las Vegas Sands Corp.,Technology,LinkedIn,https://www.linkedin.com/jobs/view/sr-business...,2026-09-16,2026-09-15,...,1,1,0,0,0,0,0,0.0,Job Description: Position Overview The primary...,A


In [3]:
# 查看薪资基本统计量
print(df[['Salary_Min', 'Salary_Max']].describe())

# 查看技能标签的平均值（即“要求该技能的比例”）
print("\n各技能要求比例：")
# 注意：把 'AI_tools' 改成了 'ai_skill_explicit'
print(df[['Excel', 'SQL', 'Python', 'R', 'Tableau', 'Power_BI', 'ai_skill_explicit']].mean())

          Salary_Min    Salary_Max
count      55.000000  5.500000e+01
mean    80476.181636  1.432290e+05
std     24944.675627  1.966633e+05
min     42000.000000  5.000000e+04
25%     63016.000000  9.055300e+04
50%     77573.000000  1.145030e+05
75%    100000.000000  1.365000e+05
max    146670.000000  1.550000e+06

各技能要求比例：
Excel                0.224490
SQL                  0.209184
Python               0.020408
R                    0.000000
Tableau              0.030612
Power_BI             0.096939
ai_skill_explicit    0.229592
dtype: float64
